# State in LangGraph

In LangGraph, **state is the shared data object that moves through the entire graph**.

Every node can:

1. Read the current state.
2. Perform some logic.
3. Return a partial update.
4. Pass the updated state to the next node.

LangGraph defines state using a schema such as `TypedDict`, `dataclass`, or Pydantic model. Each node usually receives the state as input and returns only the fields it wants to update. ([docs.langchain.com](https://docs.langchain.com/oss/python/langgraph/graph-api?utm_source=chatgpt.com))

---

## Simple definition

> State is the agent’s current working information at a particular point in execution.

For example, a customer-support agent may store:


In [ ]:
{
    "user_query": "Where is my order?",
    "order_id": "ORD101",
    "order_status": "Delayed",
    "next_action": "check_refund_policy"
}


As different nodes execute, these values are added or changed.

---

# LangGraph Example

Let us build a simple order-support workflow:


```text
START
  ↓
extract_order_id
  ↓
check_order_status
  ↓
generate_response
  ↓
END
```

## Complete code


In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


# ==========================================================
# 1. Define State Schema
# ==========================================================

class OrderState(TypedDict, total=False):
    user_query: str
    order_id: str
    order_status: str
    final_response: str


# ==========================================================
# 2. Define Nodes
# ==========================================================

def extract_order_id(state: OrderState) -> dict:
    print("State received by extract_order_id:")
    print(state)

    # For demonstration, assume we extracted this from the query
    return {
        "order_id": "ORD101"
    }


def check_order_status(state: OrderState) -> dict:
    print("State received by check_order_status:")
    print(state)

    order_id = state["order_id"]

    # Simulated database/API call
    if order_id == "ORD101":
        status = "Delayed"
    else:
        status = "Unknown"

    return {
        "order_status": status
    }


def generate_response(state: OrderState) -> dict:
    print("State received by generate_response:")
    print(state)

    order_id = state["order_id"]
    status = state["order_status"]

    response = (
        f"Your order {order_id} is currently {status.lower()}."
    )

    return {
        "final_response": response
    }


# ==========================================================
# 3. Build Graph
# ==========================================================

builder = StateGraph(OrderState)

builder.add_node("extract_order_id", extract_order_id)
builder.add_node("check_order_status", check_order_status)
builder.add_node("generate_response", generate_response)

builder.add_edge(START, "extract_order_id")
builder.add_edge("extract_order_id", "check_order_status")
builder.add_edge("check_order_status", "generate_response")
builder.add_edge("generate_response", END)

graph = builder.compile()


# ==========================================================
# 4. Invoke Graph
# ==========================================================

initial_state = {
    "user_query": "Where is my order ORD101?"
}

result = graph.invoke(initial_state)

print("\nFinal state:")
print(result)


Expected final state:


In [ ]:
{
    "user_query": "Where is my order ORD101?",
    "order_id": "ORD101",
    "order_status": "Delayed",
    "final_response": "Your order ORD101 is currently delayed."
}


---

# How State Changes Step by Step

## Initial state

When the graph starts:


In [ ]:
{
    "user_query": "Where is my order ORD101?"
}


Only `user_query` is available.

---

## Node 1: `extract_order_id`

This node receives:


In [ ]:
{
    "user_query": "Where is my order ORD101?"
}


It returns:


In [ ]:
{
    "order_id": "ORD101"
}


Notice that the node does **not** return the full state.

LangGraph merges this update with the existing state:


In [ ]:
{
    "user_query": "Where is my order ORD101?",
    "order_id": "ORD101"
}


Nodes can return partial state updates; LangGraph applies those updates to the graph state. ([docs.langchain.com](https://docs.langchain.com/oss/python/langgraph/graph-api?utm_source=chatgpt.com))

---

## Node 2: `check_order_status`

This node now receives:


In [ ]:
{
    "user_query": "Where is my order ORD101?",
    "order_id": "ORD101"
}


It returns:


In [ ]:
{
    "order_status": "Delayed"
}


Updated state:


In [ ]:
{
    "user_query": "Where is my order ORD101?",
    "order_id": "ORD101",
    "order_status": "Delayed"
}


---

## Node 3: `generate_response`

This node receives all information collected so far:


In [ ]:
{
    "user_query": "Where is my order ORD101?",
    "order_id": "ORD101",
    "order_status": "Delayed"
}


It returns:


In [ ]:
{
    "final_response": "Your order ORD101 is currently delayed."
}


Final state:


In [ ]:
{
    "user_query": "Where is my order ORD101?",
    "order_id": "ORD101",
    "order_status": "Delayed",
    "final_response": "Your order ORD101 is currently delayed."
}


---

# Important Rule: Nodes Return Updates

A common beginner mistake is returning the complete state manually:


In [ ]:
def check_order_status(state):
    state["order_status"] = "Delayed"
    return state


This may work in simple cases, but directly mutating the input state is not the recommended pattern.

Prefer:


In [ ]:
def check_order_status(state):
    return {
        "order_status": "Delayed"
    }


Think of a node as:

```text
Current state → Node logic → State update
```

Not:

```text
Current state → Manually modify everything
```

---

# State Schema

This part defines which values can exist:


In [ ]:
class OrderState(TypedDict, total=False):
    user_query: str
    order_id: str
    order_status: str
    final_response: str


Here:

- `user_query` stores the original request.
- `order_id` stores the extracted ID.
- `order_status` stores the API result.
- `final_response` stores the final answer.

`total=False` means every key is optional from Python type-checking’s perspective. This is useful because the initial state may contain only some fields, while other fields are populated later.

Without it:


In [ ]:
class OrderState(TypedDict):
    user_query: str
    order_id: str
    order_status: str


Type checkers expect all keys to be present whenever an `OrderState` object is created.

---

# Overwriting State Values

By default, when a node returns a value for an existing key, it **replaces** the old value.

Example:

Initial state:


In [ ]:
{
    "status": "Processing"
}


Node returns:


In [ ]:
{
    "status": "Completed"
}


New state:


In [ ]:
{
    "status": "Completed"
}


When no reducer is specified, LangGraph uses overwrite behavior for that state field. ([docs.langchain.com](https://docs.langchain.com/oss/python/langgraph/graph-api?utm_source=chatgpt.com))

---

# Reducers in State

Suppose we want to preserve a history of all completed steps.


In [ ]:
class AgentState(TypedDict):
    steps: list[str]


Node 1 returns:


In [ ]:
{"steps": ["Extracted order ID"]}


Node 2 returns:


In [ ]:
{"steps": ["Checked order status"]}


Without a reducer, the second update replaces the first:


In [ ]:
{
    "steps": ["Checked order status"]
}


To append instead of replace, define a reducer.


In [ ]:
from typing_extensions import TypedDict, Annotated
from operator import add


class AgentState(TypedDict):
    steps: Annotated[list[str], add]


Now the updates are combined:


In [ ]:
{
    "steps": [
        "Extracted order ID",
        "Checked order status"
    ]
}


Reducers define how updates to an individual state field are combined. Without a reducer, values overwrite; with `operator.add`, list updates are appended. ([docs.langchain.com](https://docs.langchain.com/oss/python/langgraph/use-graph-api?utm_source=chatgpt.com))

---

# Full Reducer Example


In [ ]:
from operator import add
from typing_extensions import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END


class AgentState(TypedDict, total=False):
    user_query: str
    order_id: str
    order_status: str

    # New list values will be appended
    steps: Annotated[list[str], add]


def extract_order(state: AgentState) -> dict:
    return {
        "order_id": "ORD101",
        "steps": ["Order ID extracted"]
    }


def check_status(state: AgentState) -> dict:
    return {
        "order_status": "Delayed",
        "steps": ["Order status checked"]
    }


def finish(state: AgentState) -> dict:
    return {
        "steps": ["Workflow completed"]
    }


builder = StateGraph(AgentState)

builder.add_node("extract_order", extract_order)
builder.add_node("check_status", check_status)
builder.add_node("finish", finish)

builder.add_edge(START, "extract_order")
builder.add_edge("extract_order", "check_status")
builder.add_edge("check_status", "finish")
builder.add_edge("finish", END)

graph = builder.compile()

result = graph.invoke({
    "user_query": "Where is ORD101?",
    "steps": []
})

print(result)


Output:


In [ ]:
{
    "user_query": "Where is ORD101?",
    "order_id": "ORD101",
    "order_status": "Delayed",
    "steps": [
        "Order ID extracted",
        "Order status checked",
        "Workflow completed"
    ]
}


---

# State With LLM Messages

In conversational agents, the most common state field is `messages`.


In [ ]:
from typing_extensions import TypedDict, Annotated
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages


class ChatState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]


The `add_messages` reducer tells LangGraph to merge new messages into the existing conversation instead of replacing the entire message list.


In [ ]:
def chatbot_node(state: ChatState):
    response = llm.invoke(state["messages"])

    return {
        "messages": [response]
    }


Suppose the state initially contains:


In [ ]:
{
    "messages": [
        HumanMessage(content="What is LangGraph?")
    ]
}


The node returns:


In [ ]:
{
    "messages": [
        AIMessage(content="LangGraph is a framework...")
    ]
}


After applying `add_messages`, the state becomes:


In [ ]:
{
    "messages": [
        HumanMessage(content="What is LangGraph?"),
        AIMessage(content="LangGraph is a framework...")
    ]
}


---

# Why `add_messages` Instead of `operator.add`?

Both can append messages, but `add_messages` is designed specifically for message objects.


In [ ]:
messages: Annotated[list[AnyMessage], add_messages]


It can intelligently handle message updates using message IDs. For example, an existing message with the same ID can be updated rather than blindly duplicated.

For normal lists:


In [ ]:
steps: Annotated[list[str], operator.add]


For conversation messages:


In [ ]:
messages: Annotated[list[AnyMessage], add_messages]


---

# State and Conditional Routing

State is also used to decide which node should execute next.


In [ ]:
from typing import Literal


class SupportState(TypedDict, total=False):
    order_status: str
    refund_required: bool


def route_order(state: SupportState) -> Literal[
    "refund_node",
    "status_node"
]:
    if state.get("refund_required"):
        return "refund_node"

    return "status_node"


Add the conditional edge:


In [ ]:
builder.add_conditional_edges(
    "check_order",
    route_order,
    {
        "refund_node": "refund_node",
        "status_node": "status_node"
    }
)


Flow:

```text
check_order
     ↓
Read refund_required from state
     ↓
 ┌───────────────┐
True            False
 ↓                ↓
refund_node    status_node
```

So state does not only store data. It can also control the graph’s execution path.

---

# State vs Memory

These terms are related but not identical.

## State

Current information used during graph execution:


In [ ]:
{
    "current_question": "...",
    "tool_result": "...",
    "next_action": "..."
}


## Memory

Information preserved for future use.

For example:


In [ ]:
{
    "user_preference": "Prefers concise answers"
}


A useful distinction:

```text
State = what the current workflow knows
Memory = what is retained across steps or conversations
```

State can become persistent memory when LangGraph uses a checkpointer. LangGraph’s persistence layer saves graph state as checkpoints, commonly associated with a thread. ([docs.langchain.com](https://docs.langchain.com/oss/python/langgraph/persistence?utm_source=chatgpt.com))

---

# Persistent State With a Checkpointer

Without a checkpointer:


In [ ]:
graph.invoke(...)


The state exists during that execution, and you receive the final result.

With a checkpointer, LangGraph can save the state for a conversation thread.


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

graph = builder.compile(
    checkpointer=checkpointer
)


Invoke it with a thread ID:


In [ ]:
config = {
    "configurable": {
        "thread_id": "user-101"
    }
}

result = graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "My name is Shrawan."
            }
        ]
    },
    config=config
)


The next call with the same thread ID can continue using the saved thread state.


In [ ]:
result = graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is my name?"
            }
        ]
    },
    config=config
)


Conceptually:

```text
thread_id = user-101
        ↓
Load previous checkpoint
        ↓
Apply new input
        ↓
Run graph
        ↓
Save updated checkpoint
```

For production, an in-memory checkpointer is mainly useful for development; persistent database-backed checkpointers are normally used when state must survive process restarts.

---

# State in Parallel Nodes

Suppose two nodes execute in parallel:

```text
          START
         /     \
        ↓       ↓
 search_web   search_db
         \     /
          ↓   ↓
          combine
```

Both nodes may update state.


In [ ]:
class ResearchState(TypedDict):
    results: Annotated[list[str], add]


Web node:


In [ ]:
def search_web(state):
    return {
        "results": ["Web result"]
    }


Database node:


In [ ]:
def search_db(state):
    return {
        "results": ["Database result"]
    }


Because `results` has an append reducer, LangGraph can combine both updates:


In [ ]:
{
    "results": [
        "Web result",
        "Database result"
    ]
}


Without an appropriate reducer, multiple parallel nodes updating the same key can create a concurrent-update error. ([docs.langchain.com](https://docs.langchain.com/oss/python/langgraph/errors/INVALID_CONCURRENT_GRAPH_UPDATE?utm_source=chatgpt.com))

---

# Best Practices

### Keep the state structured

Prefer:


In [ ]:
class State(TypedDict):
    user_query: str
    order_id: str
    status: str


Instead of:


In [ ]:
class State(TypedDict):
    everything: dict


### Return only changed fields


In [ ]:
return {"status": "Completed"}


### Use reducers only where accumulation is required


In [ ]:
steps: Annotated[list[str], add]


### Do not store unnecessary large objects

Avoid placing huge documents or duplicated data into every state checkpoint. Store references or only the relevant extracted information when possible.

### Separate temporary and final fields


In [ ]:
class State(TypedDict, total=False):
    user_query: str
    retrieved_documents: list[str]
    draft_answer: str
    final_answer: str


---

# Interview Answer

> In LangGraph, state is a shared structured object that flows through all nodes in the graph. It represents everything the workflow currently knows, such as user input, messages, tool outputs, intermediate results and routing decisions. Each node receives the current state and returns a partial update rather than the entire object. LangGraph merges that update into the existing state. By default, new values overwrite old values, but reducers such as `operator.add` or `add_messages` can accumulate updates. State can also be persisted through checkpointers and used for conditional routing between nodes.

## Remember

```text
State schema → Defines available fields
Node input   → Reads current state
Node output  → Returns partial update
Reducer      → Controls merge behaviour
Checkpointer → Persists state
Router       → Uses state to choose next node
```
